In [1]:
# Model agnostic 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
from typing import Optional, List, Callable, Dict, Any, List
from pathlib import Path
from utils import Helm  # custom model for data handling/model trianing

# Model specific 
from xgboost import XGBRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from typing import Optional, List 

from sklearn.linear_model import LinearRegression

This version of Feyn and the QLattice is available for academic, personal, and non-commercial use. By using the community version of this software you agree to the terms and conditions which can be found at https://abzu.ai/eula.

In [2]:
# Get the directory this file lives in
nb_dir = Path.cwd() # notebook directory
project_root = nb_dir.parents[0] # project directory
data_path = project_root / "datasets" / "processed_well_data.csv"

includ_cols = ['Dia', 'Dev(deg)','Area (m2)', 'z','GasDens','LiquidDens', 'P/T','friction_factor', 'critical_film_thickness']
D = Helm(path=data_path, includ_cols=includ_cols, test_size=0.20)

In [3]:
# define xgboost pipeline
def xgboost(
        hparams: Dict[str,Any]
) -> Pipeline:
    
    xgb = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        importance_type="gain", 
        **hparams, # ** unravels to sets of key, value 
        )
    # 2) Wrap it in SelectFromModel
    selector = SelectFromModel(
        estimator=xgb,
        threshold="mean",                # keep features with importance ≥ mean importance
        prefit=False                     # will fit selector inside the pipeline
    )

    # 3) Build a pipeline
    pipe = Pipeline([
        ("feature_sel", selector),
        ("model",       xgb),
    ])

    return pipe

hparam_grid = {
            "n_estimators":   [25, 40, 50],
            "learning_rate":  [0.01, 0.05, 0.1],
            "max_depth":      [10, 15, 50],
            "interval":       [2000],  # Classification tolerance for well status
        }
# train model and optimize hyperparameters via grid search 
trained_model = D.evolv_model(build_model=xgboost, hparam_grid=hparam_grid, k_folds=5)

mask = trained_model.named_steps["feature_sel"].get_support()  
#    ↑ this is a 1d array of True/False of length n_features

# index into column names
selected_features = D.X.columns[mask]

print("Features kept by SelectFromModel:")
print(selected_features.tolist())

Training model and optimizing hyperparameters via k-fold CV...


Features kept by SelectFromModel:
['Area (m2)']


Retraining optimized model on full training set
Best CV Classification Accuracy = 0.5725 ± 0.0455
Best Hyperparameters: {'n_estimators': 40, 'learning_rate': 0.1, 'max_depth': 10, 'interval': 2000}
Training Regression Metrics: RMSE=81267.2719, MAE=64989.0426, R2=0.5824
Test Regression Metrics: RMSE=88466.9344, MAE=72736.2578, R2=0.4722
Test Classification Accuracy = 0.6190


In [4]:
import pandas as pd


y_pred_scaled = trained_model.predict(D.X_scaled)
y_pred = D.scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()

# Assuming y_pred and x1 are 1D arrays or lists of equal length
df = pd.DataFrame({
    'GasFlowrate': D.gsflow,
    'y_pred': y_pred
})

df.to_csv('output_xgboost.csv', index=False)

AttributeError: 'Helm' object has no attribute 'X_scaled'